In [ ]:
!pip install xgboost imbalanced-learn scikit-learn pandas matplotlib seaborn -q
print("✅ Librerías instaladas")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
print("✅ Librerías listas")

In [ ]:
np.random.seed(42)
X_data, y_data = make_classification(
    n_samples=284807,
    n_features=29,
    n_informative=20,
    n_redundant=4,
    n_clusters_per_class=1,
    weights=[0.9928, 0.0072],
    random_state=42
)
columnas = [f'V{i}' for i in range(1, 29)] + ['Amount']
df = pd.DataFrame(X_data, columns=columnas)
df['Class'] = y_data
print(f"✅ Dataset generado: {df.shape[0]:,} filas")
print(f"Fraude: {df['Class'].sum():,} ({df['Class'].mean()*100:.2f}%)")

In [ ]:
X = df.drop('Class', axis=1)
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"✅ Train: {X_train.shape[0]:,} filas | Test: {X_test.shape[0]:,} filas")
print(f"Fraude en train: {y_train.sum():,} | Fraude en test: {y_test.sum():,}")

In [ ]:
print("⏳ Aplicando SMOTE...")
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)
print(f"✅ Datos balanceados")
print(f"Antes  — Normal: {y_train.value_counts()[0]:,} | Fraude: {y_train.value_counts()[1]:,}")
print(f"Después — Normal: {y_train_bal.value_counts()[0]:,} | Fraude: {y_train_bal.value_counts()[1]:,}")

In [ ]:
from sklearn.ensemble import RandomForestClassifier
import time

print("🌲 Entrenando Random Forest...")
start = time.time()

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train_bal, y_train_bal)

print(f"✅ Random Forest entrenado en {time.time()-start:.1f} segundos")

In [ ]:
from xgboost import XGBClassifier

print("⚡ Entrenando XGBoost...")
start = time.time()

xgb_model = XGBClassifier(
    n_estimators=100,
    random_state=42,
    eval_metric='logloss',
    use_label_encoder=False
)
xgb_model.fit(X_train_bal, y_train_bal)

print(f"✅ XGBoost entrenado en {time.time()-start:.1f} segundos")

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

# Predicciones
rf_pred = rf_model.predict(X_test)
xgb_pred = xgb_model.predict(X_test)

rf_proba = rf_model.predict_proba(X_test)[:, 1]
xgb_proba = xgb_model.predict_proba(X_test)[:, 1]

# AUC-ROC
rf_auc = roc_auc_score(y_test, rf_proba)
xgb_auc = roc_auc_score(y_test, xgb_proba)

print("=" * 50)
print(f"📊 AUC-ROC Random Forest : {rf_auc:.4f}")
print(f"📊 AUC-ROC XGBoost       : {xgb_auc:.4f}")
print("=" * 50)

print("\n🌲 Classification Report — Random Forest:")
print(classification_report(y_test, rf_pred, target_names=["Normal", "Fraude"]))

print("\n⚡ Classification Report — XGBoost:")
print(classification_report(y_test, xgb_pred, target_names=["Normal", "Fraude"]))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, model_pred, title in zip(
    axes,
    [rf_pred, xgb_pred],
    ["Random Forest", "XGBoost"]
):
    cm = confusion_matrix(y_test, model_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=["Normal", "Fraude"],
                yticklabels=["Normal", "Fraude"])
    ax.set_title(f"Matriz de Confusión — {title}", fontsize=13, fontweight='bold')
    ax.set_ylabel("Real")
    ax.set_xlabel("Predicción")

plt.tight_layout()
plt.savefig("matrices_confusion.png", dpi=150, bbox_inches='tight')
plt.show()
print("✅ Gráfico guardado: matrices_confusion.png")

In [ ]:
from sklearn.metrics import roc_curve

fpr_rf, tpr_rf, _ = roc_curve(y_test, rf_proba)
fpr_xgb, tpr_xgb, _ = roc_curve(y_test, xgb_proba)

plt.figure(figsize=(9, 6))
plt.plot(fpr_rf, tpr_rf, label=f"Random Forest (AUC = {rf_auc:.4f})", color='steelblue', lw=2)
plt.plot(fpr_xgb, tpr_xgb, label=f"XGBoost (AUC = {xgb_auc:.4f})", color='darkorange', lw=2)
plt.plot([0, 1], [0, 1], 'k--', label="Clasificador aleatorio")
plt.xlabel("Tasa de Falsos Positivos", fontsize=12)
plt.ylabel("Tasa de Verdaderos Positivos", fontsize=12)
plt.title("Curvas ROC — Comparación de Modelos", fontsize=14, fontweight='bold')
plt.legend(loc="lower right", fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("curvas_roc.png", dpi=150, bbox_inches='tight')
plt.show()
print("✅ Gráfico guardado: curvas_roc.png")

In [ ]:
import numpy as np
from sklearn.metrics import f1_score, precision_score, recall_score

metricas = {
    'AUC-ROC':   [rf_auc, xgb_auc],
    'Precision': [precision_score(y_test, rf_pred), precision_score(y_test, xgb_pred)],
    'Recall':    [recall_score(y_test, rf_pred),    recall_score(y_test, xgb_pred)],
    'F1-Score':  [f1_score(y_test, rf_pred),        f1_score(y_test, xgb_pred)]
}

x = np.arange(len(metricas))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x - width/2, [v[0] for v in metricas.values()], width, label='Random Forest', color='steelblue')
bars2 = ax.bar(x + width/2, [v[1] for v in metricas.values()], width, label='XGBoost', color='darkorange')

ax.set_xticks(x)
ax.set_xticklabels(metricas.keys(), fontsize=12)
ax.set_ylabel("Score", fontsize=12)
ax.set_title("Comparación de Métricas — Random Forest vs XGBoost", fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.set_ylim(0, 1.1)
ax.grid(axis='y', alpha=0.3)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{bar.get_height():.3f}', ha='center', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{bar.get_height():.3f}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig("comparacion_modelos.png", dpi=150, bbox_inches='tight')
plt.show()
print("✅ Gráfico guardado: comparacion_modelos.png")

## 📝 Conclusiones

- **XGBoost superó a Random Forest** en AUC-ROC, siendo la métrica correcta para datasets desbalanceados.
- **SMOTE fue clave**: balanceó las clases de entrenamiento de 2,813 fraudes a 225,032, mejorando significativamente la detección.
- El **accuracy alto de Random Forest (≈99%) es engañoso** — refleja que predice "Normal" casi siempre.
- En detección de fraude, **el Recall es prioritario**: es más costoso no detectar un fraude que generar una alerta falsa.
- **Modelo seleccionado: XGBoost** por mayor AUC-ROC y mejor balance entre precisión y recall en la clase minoritaria.

*Proyecto desarrollado como parte del programa INDOTEL/BID/CYMETRIA 2026.*